In [ ]:
import os
import warnings
import logging
%pip install -q sktime
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CPP_MIN_VLOG_LEVEL'] = '3'
os.environ['ABSL_CPP_MIN_LOG_LEVEL'] = '3'

warnings.filterwarnings('ignore')
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

logging.getLogger('tensorflow').setLevel(logging.ERROR)
logging.getLogger('absl').setLevel(logging.ERROR)

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm.auto import tqdm
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV, GroupKFold, ParameterGrid
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.utils.class_weight import compute_class_weight
from sklearn.utils.validation import check_is_fitted
import sys
from sklearn.feature_selection import SelectKBest, f_classif, SelectPercentile

sys.path.append('/kaggle/input/datasets/keithmarange/mini-rocket-methods/')
sys.path.append('/kaggle/input/cmi-competition-code')
import utils

import tensorflow as tf
tf.get_logger().setLevel('ERROR')
tf.autograph.set_verbosity(3)

from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import ParameterSampler, RandomizedSearchCV
import importlib

from scipy.stats import randint, uniform, loguniform

tf.keras.backend.clear_session()

In [ ]:
data_folder = utils.find_data_root()

raw_train_df  = pd.read_csv(data_folder / 'train.csv')
raw_test_df   = pd.read_csv(data_folder / 'test.csv')
train_demo_df = pd.read_csv(data_folder / 'train_demographics.csv')
test_demo_df  = pd.read_csv(data_folder / 'test_demographics.csv')

temp_calculations_folder_name = 'temp_calculations/'
model_run_folder_name = 'model_runs/'
os.makedirs(temp_calculations_folder_name, exist_ok=True)
os.makedirs(model_run_folder_name, exist_ok=True)

In [ ]:
model_run_name = 'mini_rocket_v1'

feat_columns = ['sequence_type', 'sequence_id', 'sequence_counter', 'subject', 'orientation', 'behavior', 'phase', 'gesture']
acc_columns  = ['acc_x', 'acc_y', 'acc_z']
rot_columns  = ['rot_w', 'rot_x', 'rot_y', 'rot_z']
thm_columns  = ['thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5']

non_device_cols = [
    'sequence_type', 'sequence_id', 'sequence_counter', 'subject', 'orientation',
    'behavior', 'phase', 'gesture', 'adult_child', 'age', 'sex', 'handedness',
    'height_cm', 'shoulder_to_wrist_cm', 'elbow_to_wrist_cm'
]

sampling_rate = 100
pipe_name = 'temporal_extractor'

n_splits = 3
tof_columns = [f'tof_{i}_v{j}' for i in range(1, 6) for j in range(0, 64)]

orientation_cols_dict = {
    'Lie on Back': ['Lie on Back'],
}

# orientation_cols_dict = {
#     'All': ['Seated Straight', 'Lie on Side - Non Dominant', 'Seated Lean Non Dom - FACE DOWN', 'Lie on Back'],
#     'Lie on Back': ['Lie on Back'],
# }

model_target_list = ['gesture_action']

do_report    = False
save_model   = False
random_search = False
train_size = 0.2

select_k_name = 'select_k_percentile'

In [ ]:
train_df = raw_train_df.set_index('row_id')
target_only_df = train_df[train_df['sequence_type'] == 'Target'].copy()

target_only_df['gesture_position'] = target_only_df['gesture'].str.split(' - ').str[0]
target_only_df['gesture_action']   = target_only_df['gesture'].str.split(' - ').str[-1]
target_only_df = target_only_df[target_only_df['phase'] == 'Gesture']

train_sample_df, test_sample_df = utils.sample_balanced_split(
    target_only_df,
    train_pct=train_size,
    test_pct=0.2
)

In [ ]:
if random_search:
    param_grid = {
        # RawSequenceExtractor (same as before)
        f'{pipe_name}__acc_cols':      [acc_columns],
        f'{pipe_name}__rot_cols':      [rot_columns],
        f'{pipe_name}__thm_cols':      [thm_columns],
        f'{pipe_name}__tof_cols':      [None],
        f'{pipe_name}__acc_mode':      ['acceleration'],
        f'{pipe_name}__rotation_mode': ['quaternion'],
        f'{pipe_name}__thm_mode':      ['delta'],
        f'{pipe_name}__tof_mode':      ['baseline'],

        f'{select_k_name}__k': randint(1, 100),

        'sequence_builder__maxlen':         [55],  # Integer range
        'sequence_builder__padding_value':  [-999.0],

        'classifier__estimator__num_kernels': randint(1,2000),
        'classifier__estimator__random_state': [42],
    }
else:
    param_grid = {
        # RawSequenceExtractor
        f'{pipe_name}__acc_cols':      [acc_columns],
        f'{pipe_name}__rot_cols':      [rot_columns],
        f'{pipe_name}__thm_cols':      [thm_columns],
        f'{pipe_name}__tof_cols':      [tof_columns],
        f'{pipe_name}__acc_mode':      ['acceleration'],
        f'{pipe_name}__rotation_mode': ['quaternion'],
        f'{pipe_name}__thm_mode':      [ 'delta'],
        f'{pipe_name}__tof_mode':      ['raw', 'delta', 'centered', 'baseline'],

        f'{select_k_name}__percentile': [100],

        'sequence_builder__maxlen':         [60],
        'sequence_builder__padding_value':  [-999.0],

        # ============ MiniROCKET Fixed Parameters ============
        'classifier__estimator__num_kernels': [2000],
        'classifier__estimator__random_state': [42],
    }

In [ ]:
importlib.reload(utils)

raw_extractor = utils.RawSequenceExtractor(
    acc_cols=acc_columns,
)

preprocessor = ColumnTransformer([
    ('scale', StandardScaler(), make_column_selector(pattern='acc|rot|thm|tof')),
], remainder='drop', verbose_feature_names_out=False)
preprocessor.set_output(transform='pandas')

feature_selector = SelectKBest(score_func=f_classif, k=2)

sequence_builder = utils.SequencePadder(maxlen=110, padding_value=-999.0)
rocket_clf = utils.MiniRocketClassifier(num_kernels=1000, random_state=42)
classifier = utils.ManyToOneWrapperRNN(estimator=rocket_clf, target='gesture_action')

pipeline = Pipeline([
    (pipe_name, raw_extractor),
    ('preprocessor', preprocessor),
    ('feature_selector', feature_selector),
    ('sequence_builder', sequence_builder),
    ('classifier', classifier),
])

cv = GroupKFold(n_splits=n_splits)

In [ ]:
for model_target in model_target_list:

    cv_results_list = []
    for col in orientation_cols_dict:
        if random_search:
            search_obj = RandomizedSearchCV(
                estimator=pipeline,
                param_distributions=param_grid,
                n_iter=3,
                cv=cv,
                random_state=42,
                n_jobs=1,
                verbose=3,
                return_train_score=True
            )
        else:
            search_obj = GridSearchCV(
                estimator=pipeline,
                param_grid=param_grid,
                cv=cv,
                verbose=3,
                n_jobs=1,
                return_train_score=True
            )

        sliced_data_df = train_sample_df[train_sample_df['orientation'].isin(orientation_cols_dict[col])]
        y = sliced_data_df[['sequence_id', model_target]]
        groups = sliced_data_df['sequence_id']

        search_obj.fit(sliced_data_df, y, groups=groups)

        if save_model:
            model = search_obj.best_estimator_
            path_to_model_run_name = model_run_folder_name + f'{model_run_name}_{col}_{model_target}.pkl'
            joblib.dump(model, path_to_model_run_name)

        cv_results_df = pd.DataFrame(search_obj.cv_results_)
        cv_results_df['orientation_data'] = col
        cv_results_list.append(cv_results_df)

    master_cv_results_df = pd.concat(cv_results_list)
    master_cv_results_df['model_target'] = model_target
    master_cv_results_df['train_size'] = train_size
    master_cv_results_df['is_random_search'] = random_search
    path_to_cv_results = model_run_folder_name + f'{model_run_name}_{model_target}_results.csv'
    master_cv_results_df.to_csv(path_to_cv_results, index=False)